## gender

In [2]:
import pandas as pd
import re
import json

c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.21-gcc_8_3_0.dll
c:\Users\sumse\miniconda3\envs\data_quality\lib\site-packages\numpy\.libs\libopenblas_v0.3.23-gcc_8_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


In [3]:
# import dictionary of education(json)

with open(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\gender\gender_pattern.json",encoding='utf-8')as fin:
    pattern_dict = json.load(fin)

In [4]:
# import data

data_gender = pd.read_csv(r"D:\Portfolio\Data Engineer Intern at BTS Group (Jul - Nov 2023)\Data Quality Improvement Project\notebook_python\gender\1_customer_comp_gender.csv" , encoding = 'utf8')

data_gender.shape

(42, 7)

In [5]:
pd.set_option('display.max_rows', None)

In [6]:
data_gender.head(5)

,subject,no,id,case,detail case,flag,gender
0,gender,1,gender-ENG-1,ENG,normal,good,M
1,gender,2,gender-ENG-2,ENG,normal,good,F
2,gender,3,gender-ENG-3,ENG,normal,good,male
3,gender,4,gender-ENG-4,ENG,normal,good,female
4,gender,5,gender-ENG-5,ENG,spell,bad,mm


In [7]:
data_gender.tail(5)

,subject,no,id,case,detail case,flag,gender
37,gender,38,gender-THA-38,THA,spell,bad,หยิน
38,gender,39,gender-THA-39,THA,spell,bad,maleชา
39,gender,40,gender-THA-40,THA,spell,bad,maleชา
40,gender,41,gender-THA-41,THA,spell,bad,femaleห
41,gender,42,gender-THA-42,THA,spell,bad,หญิ


In [8]:
# make the data to lower case
# remove .
# Consecutively characters convert to one-character
def standard_transform(text_input):
    text_input = text_input.lower().replace('.' , '')
    text_input = re.sub(r'([a-zก-๛])\1+' , r'\1' , text_input)
    return text_input



# fransform data
data_gender['gender_transform'] = data_gender['gender'].apply(lambda raw_data : standard_transform(raw_data))

In [9]:
# choose only english letters
def take_eng(input_gender) :
    try :
        result = re.search(r'[a-zA-Z]+' , input_gender).group()    
    except AttributeError :
        return None
    return result



# create column for english letters
data_gender['df_eng'] = data_gender['gender_transform'].apply(lambda gender_transform : take_eng(gender_transform))

In [10]:
# choose only thai letters
def take_thai(input_gender) :
    try :
        result = re.search(r'[ก-๛]+' , input_gender).group()
    except AttributeError :
        return None
    return result



# create column for thai letters
data_gender['df_thai'] = data_gender['gender_transform'].apply(lambda gender_transform : take_thai(gender_transform))

In [11]:
# create data frame for case = eng and case = thai
gender_eng = data_gender[data_gender['df_eng'].notnull()]
gender_thai = data_gender[data_gender['df_thai'].notnull()]

# del original data frame
del data_gender

In [12]:
# create standard format
def std_gend(text_input,lang) :
    try :
        for pattern,value in pattern_dict[lang].items():
            if re.search(pattern , text_input):
                return value
            else:
                continue
        return None
    except :
        return None
    
    
    
    
# create column for standard format
gender_eng['standard_gender'] = gender_eng['df_eng'].apply(lambda df_eng : std_gend(df_eng , 'eng'))
gender_thai['standard_gender'] = gender_thai['df_thai'].apply(lambda df_thai : std_gend(df_thai , 'tha'))

In [13]:
# combined data frame of english and thai
gender = pd.concat([gender_eng , gender_thai])

In [14]:
# standard output / status
def result(text_input, standard):
    if isinstance(text_input , str) and standard == None :
        return text_input , 0
    
    elif text_input == standard :
        return text_input , 1
    
    elif text_input != standard :
        return standard , 2
    
    elif text_input == None and standard == None :
        return text_input , 3
    
    
gender[['std_output' , 'status']] = gender.apply(lambda row: pd.Series(result(row['gender'], row['standard_gender'])), axis=1)

In [17]:
selected_cols = ['id','gender','standard_gender','std_output' ,'status']

gender = gender[selected_cols]

In [18]:
gender.tail(5)

,id,gender,standard_gender,std_output,status
37,gender-THA-38,หยิน,female,female,2
38,gender-THA-39,maleชา,male,male,2
39,gender-THA-40,maleชา,male,male,2
40,gender-THA-41,femaleห,female,female,2
41,gender-THA-42,หญิ,female,female,2
